# MEA analysis walkthrough

This notebook provides the shared MEA workflow: ingest metadata, browse experiments, find and compare datasets for a protocol, initialize the analysis pipeline, and validate cell matching and response consistency.

**§1–§5 select a dataset. §6–§8 build and validate its pipeline.**

For protocol-specific analysis, copy this notebook, update `PROTOCOL_SEARCH` in §4, and continue after §8 with the relevant condition grouping and response analysis. The shared sections intentionally do not interpret stimulus parameters.

## 1. Imports and setup

`retinanalysis` resolves its public API lazily, so this cell stays cheap — DataJoint is only pulled in when the first database-backed function is called, in §2.

In [2]:
import retinanalysis as ra
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

# Cell types used throughout. Defined here rather than in §5 because §6 and §7
# need them too, and §5 is optional — you skip it once you know which dataset
# you want, and a constant defined there would leave the rest NameError-ing.
#
#   cell_types — everything worth drawing a mosaic for.
#   MAIN_TYPES — the subset carried into the per-dataset views, where one
#                panel or line per type has to stay readable.
cell_types = ['OnP', 'OffP', 'OnM', 'OffM', 'A2', 'OnS', 'OffS']
MAIN_TYPES = ['OnP', 'OffP', 'OnM', 'OffM']

## 2. Populate the database

Ingests every experiment found on the configured source volumes into DataJoint, and re-ingests any date whose meta/tags `.json` has been modified since it was added (source mtime vs `Experiment.date_added`).

Volumes are swept in read-priority order — **ChrisNewSSD → ChrisProSSD → NAS** — and the first drive holding a given date wins, so a duplicate copy sitting on a slower volume never re-triggers an ingest. `ra.ingest_source_dirs()` reports the `(h5, meta, tags)` triples that will actually be searched; a drive that isn't mounted silently drops out of the list. The order itself lives in `src/retinanalysis/config/config.ini`, one section per volume.

Safe to re-run: dates already in the database and unchanged on disk are skipped.

In [3]:
# Volumes that will be swept, in read-priority order (local SSDs before the NAS).
for h5_dir, meta_dir, tags_dir in ra.ingest_source_dirs():
    print(f'h5   : {h5_dir}\nmeta : {meta_dir}\ntags : {tags_dir}\n')

# Ingest new dates, and refresh any date whose json changed since it was added.
# Returns {'n_ingested', 'added', 'updated', 'skipped'}.
summary = ra.populate_database()

print(f"\nnewly added : {len(summary['added'])}")
print(f"refreshed   : {len(summary['updated'])}")
print(f"errored     : {len(summary['skipped'])}")

h5   : /Volumes/ChrisProSSD/data/h5
meta : /Volumes/ChrisProSSD/data/datajoint_testbed/mea/meta
tags : /Volumes/ChrisProSSD/data/datajoint_testbed/mea/tags



/Users/chrischen/opt/anaconda3/envs/retinanalysis/lib/python3.11/site-packages/datajoint/settings.py:979: UserWarning: No datajoint.json found. Using defaults and environment variables. Run `dj.config.save_template()` to create a template configuration.
  config = _create_config()


Ingest source: /Volumes/ChrisProSSD/data/h5
Skipped 190 date(s) with metadata but no sorted data on any mounted volume: 20220405C, 20220406C, 20220412C, 20220420C, 20220426C, 20220518C (+184 more)


Experiments:   0%|          | 0/31 [00:00<?, ?it/s]

Already in database: 20260113C
Already in database: 20251112C
Already in database: 20251215C
Already in database: 20251022C
Already in database: 20260102C
Already in database: 20230111C
Already in database: 20251008C
Already in database: 20250121C
Already in database: 20251006C
Already in database: 20230214C
Already in database: 20250924C
Already in database: 20250514C
Already in database: 20230228C
Already in database: 20250429C
Already in database: 20250321C
Already in database: 20230313C
Already in database: 20250306C
Already in database: 20230502C
Already in database: 20230516C
Already in database: 20230725C
Already in database: 20231026C
Already in database: 20231220C
Already in database: 20240117C
Already in database: 20240130C
Already in database: 20240523C
Already in database: 20240801C
Already in database: 20220823C
Already in database: 20221101C
Already in database: 20221123C
Already in database: 20230523C
Already in database: 20260318C

No experiments skipped due to errors.


## 3. What's in the database?

A read-only survey of everything §2 ingested, before narrowing to one protocol. The implementations live in `retinanalysis/utils/db_summary.py` so this section stays two calls.

The first cell answers three questions:

1. **How many recordings of each kind** — `ra.recording_counts()`, split by `Experiment.is_mea` into MEA arrays and single-cell patch experiments.
2. **Which species** — `ra.species_counts()`, from `Animal.species`. Recorded for most MEA dates but rarely for patch experiments, so expect a large `(not recorded)` row on the patch side; that means the field is blank, not that the animal is missing.
3. **Which protocols dominate the MEA corpus** — `ra.mea_protocol_counts()`, ranked by number of distinct dates rather than number of blocks. Dates is the more useful denominator: a protocol run four times in one day is still one day of data. Both counts are returned so the difference is visible.

The protocol table is rendered with `ra.scroll_table` rather than `display()`, so the **whole** list is there — a fixed-height box with a sticky header, scrolling only once the corpus outgrows it. It used to be truncated to `head(15)`, which hid exactly the long tail you go looking for when hunting a protocol you ran a handful of times. Raise `height=` for a taller box.

The second cell is `ra.browse_experiment_tree()` — a dropdown of every MEA date. Pick one and it renders that date's blocks as a **date → protocol → datafile** tree, indexed on those three levels so pandas blanks the repeated labels and the nesting shows: each date once, each protocol once beneath it, and the datafiles that ran it underneath. Columns are the group label, the filter-wheel reading, the sorting chunk and the block duration. `ra.experiment_tree('20231220C')` returns the same table directly when you already know the date and don't want the widget.

Protocol names are shortened to their last dotted component — `edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground` becomes `EyeMovementTrajectoryAlternatingBackground`. The prefix only records which lab package the protocol came from and makes every table unreadable.

**On the filter wheel.** `filter_wheel_ndf` is the epoch parameter `NDF`. In this codebase those are the same thing — `populate_ndf_column` calls it "the filter wheel ND being used" and the single-cell code renames the identical field to `filter_wheel_ndf`, so the tree uses the explicit name. It is read from the first epoch of each block via a JSON extraction pushed into SQL, which is why a date loads in one query instead of one per block.

In [3]:
# Whole-database survey. Each helper is one bulk query joined in pandas —
# implementations live in retinanalysis/utils/db_summary.py.
#
# recording_summary is species x rig with totals on both margins. The old
# recording_counts table was exactly this one's column sums, so showing both
# was the same numbers twice.
display(ra.recording_summary())

protocol_counts = ra.mea_protocol_counts()
print(f'{len(protocol_counts)} distinct protocols across MEA dates, '
      f'ranked by number of dates:')

# Every protocol, not a head(15) truncation: scroll_table caps the box height
# and pins the header row, so the list only scrolls once it outgrows the box.
ra.scroll_table(protocol_counts.reset_index(), height = 420,
                num_cols = ('n_dates', 'n_blocks'));

rig,MEA,patch,total
species,,,
M. nemestrina,112,4,116
M. mulatta,46,2,48
M. fascicularis,18,5,23
(not recorded),34,512,546
total,210,523,733


84 distinct protocols across MEA dates, ranked by number of dates:


protocol,n_dates,n_blocks
ContrastResponseGrating,160,764
SpatialNoise,144,639
MovingChromaticBar,124,251
ObjectMotionDots,87,154
GratingDSOS,80,107
PresentImages,72,178
ChirpStimulus,65,110
FastNoise,64,356
DovesMovie,60,127
SparseNoise,57,91


In [4]:
# Dropdown of every MEA date; picking one loads that date's block tree.
# Without the GUI, for a date you already know:  ra.experiment_tree('20231220C')
ra.browse_experiment_tree();

## 4. Find datasets that ran the protocol

**This is the one cell to change when analyzing a different protocol.** `PROTOCOL_SEARCH` is a lowercase substring match against protocol names in the database, so a specific fragment is enough.

The result is one row per epoch block. `chunk_name` is retained as the sorting label recorded in the database, but that label is often **not** a Vision noise-analysis directory: newer rows contain names such as `dynamics`, `eye_move`, or `var_mean` while their analysis lives under `chunk1`, `chunk2`, etc. Section 4 therefore resolves each row to `analysis_chunk_name` using the same rule as the pipeline: noise recorded before the protocol is preferred, candidates are ordered by time distance, and the first candidate with both a Kilosort directory and a classification file wins.

`ss_version` belongs to that resolved analysis chunk. Detection searches every mounted tier and prefers `kilosort2.5`, then `kilosort2`, then `kilosort4`. Thus `not found` now means no usable typed noise chunk was found—not merely that the database sorting label was not a directory.

Dates with no analysis directory on any currently mounted volume are removed and printed. The ID columns remain on `exp_search`; the display is narrowed to `SEARCH_COLS`. Every downstream selection uses the table's row index.


In [ ]:
PROTOCOL_SEARCH = 'EyeMovementTrajectoryAlternatingBackground'   # <-- EDIT ME

exp_search = ra.get_datasets_from_protocol_names(PROTOCOL_SEARCH)
print(f'\nsearch found {len(exp_search)} blocks across '
      f'{exp_search["exp_name"].nunique()} dates')

# Keep dates with an analysis tree on at least one mounted volume.
available_experiments = sorted({exp for root in ra.tier_dirs('analysis')
                                for exp in os.listdir(root)})
missing = sorted(set(exp_search.query('exp_name not in @available_experiments')
                     ['exp_name']))
if missing:
    print(f'dropped {len(missing)} date(s) with no analysis directory on a '
          f'mounted volume: {", ".join(missing)}')

exp_search = exp_search.query(
    'exp_name in @available_experiments').reset_index(drop=True)

# Resolve the actual typed noise-analysis chunk. The database chunk_name is
# kept for provenance; it is not assumed to be a directory name.
exp_search = ra.add_analysis_chunk_columns(exp_search)
loadable = (exp_search['ss_version'] != 'not found').sum()
print(f'{len(exp_search)} blocks remain across '
      f'{exp_search["exp_name"].nunique()} dates; '
      f'{loadable} have a loadable typed noise chunk\n')

SEARCH_COLS = ['exp_name', 'datafile_name', 'analysis_chunk_name',
               'ss_version', 'chunk_distance_min', 'NDF', 'group_label',
               'chunk_name']
display(exp_search[SEARCH_COLS])


## 5. Plot mosaics for candidate datasets

RF mosaics for the datasets you name, so you can judge which experiment has the cleanest typing before committing to one. Set `MOSAIC_ENTRIES` to row indices from the §4 table — pick rows whose `ss_version` isn't `not found`, since those have no usable typed noise chunk. Those same row indices are what §6 takes to build the pipeline, so whichever mosaic wins here you carry forward by its number.

The second cell says what is actually *in* each mosaic, which the pictures alone won't tell you. It's a **dropdown, not a loop**: `ra.browse_chunk_summaries(chunks)` shows one chunk at a time, so the cell's output stays the same length whether you loaded three datasets or fifteen. Each chunk renders on first selection and is then cached as an image, so revisiting one is instant and a chunk you never open costs nothing. The dropdown label carries the sort version and how many cells cleared `minimum_n` — usually enough to skip the thin ones without opening them.

For the selected chunk you get a table from `cell_type_summary`:

- **Cells per type.** A mosaic can look tidy while resting on four cells. It also lists the types you didn't ask for (`Unknown`, `OffMystery`, …), which is useful context — on `20231220C/chunk4` the largest single group is 286 `Unknown` cells against 116 `OffP`.
- **Firing rate per type**, as mean/median/min/max. This is the mean rate across the whole noise chunk, so it's a data-quality number, not a response measure. A type whose rates sit near zero is usually a sorting artifact rather than a population.

Then the three views a spatial mosaic can't give you, one column per cell type in `MAIN_TYPES` order so the columns line up with the mosaic above:

- **Temporal RF** — the green-channel STA time course, every cell in a dim line with the mean ± SEM over the top, on a millisecond axis ending at the spike. This is where a mislabeled type shows itself: an `OnP` group whose mean filter is flat, or inverts, was never a population no matter how regular its mosaic looks. Green is the channel plotted because the noise here is achromatic — red and green are identical — and one trace per type keeps the four columns comparable. The frame interval comes from the chunk's recorded `refreshPeriod` rather than an assumed 60 Hz, because STA depth varies between sorts (30 and 61 frames both occur).
- **Autocorrelation (ISI)** — Vision's autocorrelation histogram, sum-normalized per cell so a fast-firing cell doesn't dominate the mean. Read the left edge first: density at lags under ~2 ms is a refractory-period violation, which means the cluster merges more than one unit. A clean type dips to zero at zero lag and peaks at that type's preferred interval.
- **Spike count over the noise run** — one ECDF across all types, total spikes per cell. This is the "how much data is behind each STA" view, and it's the number that matters for whether an RF fit means anything; the rate columns in the table above are the same quantity divided by chunk duration, which is what you want when comparing chunks of different length instead.

The count distribution is drawn as an **ECDF rather than a histogram**, because a well-populated type here has ~100 cells and a marginal one has three, and at those counts a histogram's shape is mostly an artifact of where the bins fell. An ECDF is exact at any n — one step per cell — so a sparse type overlaid on a dense one stays honest, and curves separate vertically instead of occluding each other. Read it as: further right means more spikes, steeper means the type is more tightly clustered. The x axis is logarithmic because counts run over orders of magnitude between types; pass `log_x=False` for a linear one.

To render every chunk inline instead — for exporting the notebook, say, where a dropdown is dead — loop `ra.plot_chunk_panels(chunk, ...)` over `chunks.items()`. `ra.plot_spike_count_distribution` and `ra.plot_firing_rate_distribution` draw the bottom panel on its own.

**The noise chunk was already resolved in §4 by recording time, preferring one recorded before the protocol.** A chunk that ran afterwards has an intervening protocol's worth of adaptation between it and the data being typed, so it is used only when no preceding typed chunk works. This section loads that exact `analysis_chunk_name`, and §6 pins the pipeline to the same table value, so the mosaic, reported Kilosort version, and analysis cannot drift onto different chunks.

Sort versions are resolved per chunk, and the analysis directory is looked up with `find_path`, which walks the volumes in read-priority order — so a chunk that only exists on the NAS is found there when ChrisProSSD doesn't have it. A chunk missing everywhere is reported and skipped rather than aborting the sweep.

In [ ]:
# <-- EDIT ME: row indices from the §4 table. Pick rows whose ss_version is not
# 'not found'; each costs one Vision analysis-chunk read.
MOSAIC_ENTRIES = [0,1,2,3,4,5]

mosaic_search = exp_search.loc[MOSAIC_ENTRIES]
display(mosaic_search[SEARCH_COLS])

# Use the already-resolved analysis_chunk_name from §4 so the mosaic, version
# label, and eventual pipeline all refer to the same noise chunk.
all_axes, chunks = ra.plot_mosaics_for_datasets(
    mosaic_search, cell_types, minimum_n=3, b_zoom=True,
    include_neurons=True, return_chunks=True, use_chunk_column=True)


In [5]:
# What is actually in each mosaic: cells per type, temporal RF, autocorrelation
# and spike counts. One chunk at a time from the dropdown, so the output stays
# the same length however many entries MOSAIC_ENTRIES has.
ra.browse_chunk_summaries(chunks, cell_types = MAIN_TYPES, minimum_n = 3);

## 6. Initialize the analysis pipeline

Pick a row from §4 and build the stimulus block, protocol response block, and typed noise `AnalysisChunk` used downstream.

**Name the dataset by row index, not by string.** `ENTRY` indexes `exp_search`, exactly like `MOSAIC_ENTRIES`. The experiment, datafile, and resolved noise chunk all come from that row, so the pipeline cannot silently choose a different chunk from the one whose Kilosort version §4 reported or whose mosaic §5 displayed.

The sorted-data tree and analysis tree can legitimately use different Kilosort versions. Therefore `ss_version` is printed for the resolved analysis chunk but is not passed as a blanket override to `create_mea_pipeline`; each tree detects its own version independently.


In [ ]:
ENTRY = 26   # <-- EDIT ME: row index from the §4 table, same indexing as MOSAIC_ENTRIES
# 0, on cells only, 1&2& 3 nicer off, barely any on cells; 4 seems decent mosaic, 5 data018 missing in SSD; 6 is not good across; 7 might have some good onM' 
# 8 decent, 9 bettr midgets some onP; 10 better off resp.  11, only 5 epochs; 12, off resp only, 13 low firing; 14 off M seems okay, 15 better off; 16 better off, some onM
# 17 & 18 not in SSD,  19 good off, ok on; 20-21 good; 22 seems inconsistent across cells; 23 good; 24 bad; 25 not in SSD

entry         = exp_search.loc[ENTRY]
exp_name      = entry['exp_name']
datafile_name = entry['datafile_name']
chunk_name    = entry['analysis_chunk_name']

if entry['ss_version'] == 'not found' or not chunk_name:
    raise ValueError(f'ENTRY {ENTRY} has no loadable typed noise chunk; pick another row')

print(f'{exp_name} / {datafile_name}  ({entry["ss_version"]} per §4)')
print(f'noise chunk: {chunk_name}\n')

# Do not pass ss_version as one blanket override: the protocol-data and
# analysis trees are detected independently and may legitimately differ.
pipeline = ra.create_mea_pipeline(
    exp_name, datafile_name, analysis_chunk_name=chunk_name)


## 7. Inspect the dataset
Each tab is a public function if you want it on its own: `ra.plot_match_qc`, `pipeline.plot_rf_comparison`, `ra.browse_epoch_rasters`, `ra.plot_epoch_spike_counts` and `ra.browse_epoch_count_heatmaps`. Use those when exporting the notebook, where tabs and dropdowns are dead.

In [16]:
# Cluster match, RFs, rasters and per-epoch counts, one tab each. Tabs build
# on first look, so re-running this after changing ENTRY in §6 is cheap until
# you actually open a tab.
pipeline.inspect(cell_types = MAIN_TYPES, minimum_n = 4);

HTML(value='<b>20251022C / data013</b> — noise chunk chunk2')

## 8. Break out the objects — and hand off

The end of the shared part. `MEAPipeline` is a thin container over three members; pulling them into their own names is what a per-protocol notebook starts from.

- `stim_block` — `df_epochs` (one row per epoch, with the protocol's parameters promoted to columns where they vary) and `d_epoch_block_params` (what was fixed for the whole block). **This is where protocol-specific analysis begins**, because this is the first object whose contents differ between protocols.
- `response_block` — `df_spike_times`, one row per cell, each holding a list of per-epoch spike-time arrays in ms, with `cell_type` and `noise_id` filled in from the cluster match.
- `analysis_chunk` — the noise-derived side: RF parameters, STAs, EIs, timecourses, autocorrelations.

**To analyze a new protocol, copy this notebook**, change the search string in §4, and continue past here with that protocol's condition axes — group epochs by the parameters that varied, average PSTHs within condition, whatever the experiment was asking. Everything above stays as written and stays shared, so a fix to the pipeline or the QC reaches every protocol at once.

Useful next steps that are already written: `ra.get_spike_xarr(response_block, cell_types=...)` for a ragged (cell × epoch) array of spike times, `ra.plot_raster_with_psth` for a raster and PSTH per type, and `ra.epoch_count_matrix(response_block, cell_type)` for the (cells × epochs) count matrix behind §7's heatmap.

In [10]:
stim_block = pipeline.stim
response_block = pipeline.resp
analysis_chunk = pipeline.analysis_chunk